# PyMOL RMSD Alignment

This notebook demonstrates pairwise structure alignment using Open-Source PyMOL. We align two `Structure` inputs and inspect the RMSD returned by PyMOL `cealign` or regular `align`.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pymol-rmsd")
display_overview("pymol-rmsd")
display_docs_section("pymol-rmsd", "Background")

# PyMOL RMSD

PyMOL RMSD performs pairwise structure alignment using Open-Source PyMOL and returns the post-alignment RMSD (`pymol-rmsd-alignment`). It supports both PyMOL `cealign` and regular `align`, which lets callers choose between CE-style structural alignment and PyMOL's sequence-aware alignment workflow.

## Background

RMSD measures the average distance between corresponding atoms after superposition. It is useful when the target fold is known and the design goal is a close geometric match to a reference structure. PyMOL's `cealign` is often useful for global structure alignment, while `align` performs sequence alignment followed by structural refinement and can be useful when residue numbering or sequence length differs.

## Available tools

In [2]:
display_available_tools("pymol-rmsd")

- **`run_pymol_rmsd_alignment()`** — Pairwise structure RMSD alignment using PyMOL cealign or align.

### `run_pymol_rmsd_alignment`

PyMOL RMSD Alignment performs pairwise structural superposition and returns post-alignment RMSD plus method-specific metrics. The tool accepts standardized `Structure` inputs, so callers can pass raw PDB/CIF content, file paths, or existing `Structure` objects.

In [3]:
from pathlib import Path

from proto_tools import PyMOLRMSDConfig, PyMOLRMSDInput, PyMOLRMSDOutput, run_pymol_rmsd_alignment
from proto_tools.tools.structure_alignment.pymol_rmsd.pymol_rmsd import example_input

In [4]:
# Display input docs
display_api_reference("pymol-rmsd", "input", "run_pymol_rmsd_alignment")

# Use the tool's canonical example input - a structure aligned against itself,
# so RMSD should be near zero.
inputs = example_input()

**Input** — `PyMOLRMSDInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `target_structure` | `proto_tools.entities.structures.structure.Structure` | required | Target/reference structure. |
| `mobile_structure` | `proto_tools.entities.structures.structure.Structure` | required | Mobile/query structure to align. |

In [5]:
# Display config docs
display_api_reference("pymol-rmsd", "config", "run_pymol_rmsd_alignment")

config = PyMOLRMSDConfig(method="cealign")

**Config** — `PyMOLRMSDConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int | None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `method` | `Literal['cealign', 'align']` | `'cealign'` | PyMOL alignment routine to use for RMSD calculation. |
| `target_selection` | `str` | `'target'` | PyMOL target selection, e.g. 'target and name CA'. |
| `mobile_selection` | `str` | `'mobile'` | PyMOL mobile selection, e.g. 'mobile and name CA'. |
| `failure_rmsd` | `float` | `999.0` | RMSD returned when PyMOL cannot align the requested structures. |

In [6]:
# Run the alignment
result = run_pymol_rmsd_alignment(inputs, config)

Running run_pymol_rmsd_alignment [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("pymol-rmsd", "output", "run_pymol_rmsd_alignment")

print(f"Method: {result.method}")
print(f"RMSD: {result.rmsd:.6f} Angstrom")
print(dict(result.metrics.items()))

**Output** — `PyMOLRMSDOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `method` | `Literal['cealign', 'align']` | required | PyMOL alignment method used. |
| `metrics` | `proto_tools.tools.structure_alignment.pymol_rmsd.pymol_rmsd.PyMOLRMSDMetrics` | `PyMOLRMSDMetrics(primary_metric='rmsd')` | RMSD alignment metrics. |

Method: cealign
RMSD: 0.000000 Angstrom
{'rmsd': 2.9200193199910854e-07, 'aligned_length': 64}


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("pymol_rmsd_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pymol_rmsd_result.json'}")

Exported to example_output/pymol_rmsd_result.json
